In [59]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from bertopic import BERTopic

df = pd.read_csv("./data/trustpilot-reviews-123k.csv")
df.head()

,category,company,description,title,review,stars
0,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Great quality dog drying robe although…,Great quality dog drying robe although had to ...,5
1,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Really prompt service,"Really prompt service, The sofa covers have no...",5
2,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Life saver,I’ve purchased first of those coats in May2020...,5
3,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Brilliant coats,Brilliant coats. Really like the limited editi...,5
4,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Great company and products,Great company and products. This is my 3rd dry...,5


In [60]:
nombre_empresa = "www.sonicdirect.co.uk"
df_empresa =df[df["company"] == nombre_empresa].copy()
print(f"Dimensiones del dataset: {df_empresa.shape}")
df_empresa.head()

Dimensiones del dataset: (100, 6)


,category,company,description,title,review,stars
32160,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,Really impressed,"Really impressed, ordered on 27th Dec,wasn't e...",5
32161,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,"Great price, fantastic delivery",We purchased a tumble drier over the Christmas...,5
32162,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,Very efficient,"Easy to order, very quick delivery time even o...",5
32163,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,Our washing machine,Our washing machine broke down two days befor...,5
32164,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,"Well informed sales person, resulted in our pu...","The sales person was well informed, however, a...",5


In [61]:
sector = df_empresa["category"].iloc[0]
print(sector)

Electronics & Technology


In [62]:
df_competencia = df[
    (df["category"] == sector) & 
    (df["company"] != nombre_empresa)
].copy()

print(f"Dimensiones del dataset de la competencia: {df_competencia.shape}")
df_competencia.head()

Dimensiones del dataset de la competencia: (5496, 6)


,category,company,description,title,review,stars
29662,Electronics & Technology,www.richersounds.com,Richer Sounds is a British home entertainment ...,Dave 55,Excellent service as always shout out to Jerom...,5
29663,Electronics & Technology,www.richersounds.com,Richer Sounds is a British home entertainment ...,Great service from Richer Sounds…,Great service from the Richer Sounds team at S...,5
29664,Electronics & Technology,www.richersounds.com,Richer Sounds is a British home entertainment ...,Wanted a TV for an 84 year old relative…,Wanted a TV for an 84 year old relative who’s ...,5
29665,Electronics & Technology,www.richersounds.com,Richer Sounds is a British home entertainment ...,IMFORMED FRIENDLY HELP FROM RICHER SOUNDS,Excellent rapport and helpfulness with your So...,5
29666,Electronics & Technology,www.richersounds.com,Richer Sounds is a British home entertainment ...,Purchase of Hi-Fi Equipment From Richer Sounds,"The Sales Assistant, Richard, was very helpful...",5


In [63]:
df_competencia[df_competencia["company"] == "www.sonicdirect.co.uk"].shape

(0, 6)

In [64]:
print("Empresa:", df_empresa.shape)
print("Competencia:", df_competencia.shape)

df_competencia["company"].nunique()

Empresa: (100, 6)
Competencia: (5496, 6)


68

In [65]:
def limpiar_texto(texto):
    texto = texto.lower()  # minúsculas
    texto = re.sub(r"[^a-zA-Z\s]", "", texto)  # quitar símbolos/números
    texto = re.sub(r"\s+", " ", texto).strip()  # quitar espacios extra
    return texto

In [66]:
df_empresa["text"] = (df_empresa["title"] + " " + df_empresa["review"]).apply(limpiar_texto)

df_competencia["text"] = (df_competencia["title"] + " " + df_competencia["review"]).apply(limpiar_texto)

In [67]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_empresa = model.encode(df_empresa["text"].tolist(), show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1926.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 4/4 [00:01<00:00,  3.54it/s]


In [68]:
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(embeddings_empresa)

df_empresa["topic"] = clusters

In [69]:
for i in range(5):
    print(f"\n--- Topic {i} ---")
    print(df_empresa[df_empresa["topic"] == i]["text"].head(5))


--- Topic 0 ---
32164    well informed sales person resulted in our pur...
32168    great product excellent tv easy set up great p...
32190    tv very happy with the tv i have recently purc...
32197    the two men that delivered th the two men that...
32218    funny how the price on my sonyoled funny how t...
Name: text, dtype: str

--- Topic 1 ---
32163    our washing machine our washing machine broke ...
32182    i had problems but they were all resolved the ...
32183    good mistakes made but sorted staff friendly s...
32186    washer dryer delivery all good even despite th...
32187    chest freezer always willing to help good cust...
Name: text, dtype: str

--- Topic 2 ---
32165    excellent delivery crew delivery guys sangdeep...
32169    first time using this company wont be the last...
32173    excellent service absolutely fantastic service...
32177    always helpful friendly and always helpful fri...
32178    lovely guy who asked if i needed help lovely g...
Name: text, dtype:

In [70]:
df_empresa.head()

,category,company,description,title,review,stars,text,topic
32160,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,Really impressed,"Really impressed, ordered on 27th Dec,wasn't e...",5,really impressed really impressed ordered on t...,4
32161,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,"Great price, fantastic delivery",We purchased a tumble drier over the Christmas...,5,great price fantastic delivery we purchased a ...,3
32162,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,Very efficient,"Easy to order, very quick delivery time even o...",5,very efficient easy to order very quick delive...,4
32163,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,Our washing machine,Our washing machine broke down two days befor...,5,our washing machine our washing machine broke ...,1
32164,Electronics & Technology,www.sonicdirect.co.uk,Sonic Direct has grown to become one of the UK...,"Well informed sales person, resulted in our pu...","The sales person was well informed, however, a...",5,well informed sales person resulted in our pur...,0


In [71]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2862.14it/s]


In [72]:
def analizar_sentimiento(textos, batch_size=32):
    resultados = sentiment_model(textos, batch_size=batch_size, truncation=True)
    labels = [r["label"] for r in resultados]
    scores = [r["score"] for r in resultados]
    return labels, scores

In [73]:
labels_emp, scores_emp = analizar_sentimiento(df_empresa["text"].tolist())

df_empresa["sentiment"] = labels_emp
df_empresa["sentiment_score"] = scores_emp

In [74]:
labels_comp, scores_comp = analizar_sentimiento(df_competencia["text"].tolist())

df_competencia["sentiment"] = labels_comp
df_competencia["sentiment_score"] = scores_comp